# Nile-Chat 12B — LoRA Fine-Tuning on Raylab WhatsApp Data

A Colab-ready adaptation of `llm_finetuning.ipynb` — same section order, same
distillation-then-LoRA-then-vLLM shape, retargeted at **Nile-Chat 12B** and the
**490 real, gate-verified, deduplicated Raylab examples** already produced locally
(465 train / 25 val).

**What you will do here:** load the already-prepared dataset from Google Drive,
LoRA fine-tune Nile-Chat 12B via LLaMA-Factory, evaluate before vs. after against
real documented baseline failures, estimate cost/throughput, serve with vLLM, and
load-test.

**What you will NOT do here:** any teacher-distillation or data-formatting work —
that's Stage 0 below, already complete on the source machine.

Runtime: **Colab, 1× A100 GPU** (Runtime → Change runtime type → A100).

### Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


## Stage 12 — Merge LoRA adapter into base model

In [1]:
# 1. Remove Colab's preinstalled torch stack (and anything from a prior manual
#    install) before vLLM brings in its own matched set — mixing sources is what
#    causes the torch/torchaudio CUDA mismatch.
!pip uninstall -y torch torchvision torchaudio vllm

# 2. uv is vLLM's own recommended installer for exactly this problem.
!pip install -qU uv

# 3. --torch-backend=auto detects the real CUDA driver on this Colab A100 runtime
#    and installs one mutually-compatible torch/torchvision/torchaudio set for it,
#    instead of resolving each package independently against PyPI's separate
#    "latest" releases. --system targets Colab's interpreter directly (uv defaults
#    to requiring a virtualenv, which Colab doesn't use).
!uv pip install --system vllm --torch-backend=auto

# 4. torchao (pulled in transitively above) breaks peft's LoRA setup below if an
#    old version is present — peft raises ImportError on torchao<0.16.0, but
#    returns cleanly if it's simply absent. Not needed here (no quantization),
#    so removed rather than upgraded.
!pip uninstall -y torchao

# 5. All five exact ranges LLaMA-Factory's own check_dependencies() enforces
#    (confirmed live from its real source, not guessed one error at a time).
#    transformers' floor here also satisfies the >=4.50.0 Gemma 3 requirement.
!pip install -qU "transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0" "datasets>=2.16.0,<=4.0.0" "accelerate>=1.3.0,<=1.15.0" "peft>=0.18.0,<=0.20.0" "trl>=0.18.0,<=0.24.0"

# 6. --no-deps: LLaMA-Factory's own setup.py can otherwise pull a loose/unpinned
#    torch requirement and silently re-upgrade it, undoing step 3's matched set.
#    peft/trl are already pinned correctly by step 5 above, so --no-deps here
#    doesn't leave them missing.
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e . --no-deps

# 7. bitsandbytes for the adamw_bnb_8bit optimizer (Stage 5) -- lets us keep
#    cutoff_len at the real full 4096 (zero truncation of any example) by cutting
#    optimizer-state memory instead of dataset context. 0.50.1 (latest, verified on
#    PyPI) declares CUDA 11.8/12/13 + torch<3,>=2.4 support, matching the
#    torch==2.13.0 pinned by step 3 above -- no new CUDA-mismatch risk.
!pip install -qU "bitsandbytes>=0.50.1"

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 100.9 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 196 packages in 1.85s
Prepared 101 packages in 47.19s
Uninstalled 14 packages in 147ms
Installed 101 packages in 268ms
 + anthropic==1.0.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.4
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.7
 + cuda-python==13.3.1
 + cuda-tile==1.5.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.2.1
 + depy

In [3]:
%%writefile /content/LLaMA-Factory/examples/merge_lora/raylab_merge.yaml
### model
model_name_or_path: MBZUAI-Paris/Nile-Chat-12B
adapter_name_or_path: /gdrive/MyDrive/raylab-finetune/models/
template: gemma
trust_remote_code: true

### export
export_dir: /content/merged_model/
export_size: 5
export_device: auto  # choices: [cpu, auto]
export_legacy_format: false

Writing /content/LLaMA-Factory/examples/merge_lora/raylab_merge.yaml


In [4]:
!cd LLaMA-Factory && llamafactory-cli export examples/merge_lora/raylab_merge.yaml

/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
config.json: 100% 934/934 [00:00<00:00, 5.29MB/s]
[INFO|configuration_utils.py:780] 2026-08-21 21:54:09,132 >> loading configuration file config.json from cache at /roo

In [6]:
from huggingface_hub import HfApi, notebook_login

# 1. تسجيل الدخول (هيظهرلك مربع صغير تحت الخلية تحطي فيه الـ Token بتاعك وتدوسي Login)
# تأكدي إن الـ Token نوعه "Write"
notebook_login()

# 2. إعداد البيانات
api = HfApi()
repo_id = "mennaharmas/raylab-nilechat-12b"

# 3. إنشاء المستودع الأول (لو مش موجود)
print("جاري إنشاء الريبو (Repository) على حسابك...")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=True)

# 4. الرفع
print("جاري رفع الموديل... (ممكن ياخد شوية وقت حسب سرعة النت في كولاب)")
api.upload_folder(
    folder_path="/content/merged_model/",
    repo_id=repo_id,
    repo_type="model"
)
print("✅ تم رفع الموديل بنجاح على Hugging Face!")

جاري إنشاء الريبو (Repository) على حسابك...
جاري رفع الموديل... (ممكن ياخد شوية وقت حسب سرعة النت في كولاب)
✅ تم رفع الموديل بنجاح على Hugging Face!


### Step 2 — Serve the merged model on Colab GPU with vLLM
*(mirrors `src/run_nilechat12b.ipynb` cells 0–14, model path swapped for the merged checkpoint above)*

Run this in a **fresh Colab runtime** (`Runtime > Restart session`) after the merge above finishes and the merged files are safely on Drive — vLLM's own install brings in a different matched `torch`/`torchvision`/`torchaudio` set than LLaMA-Factory's training environment used, exactly like `run_nilechat12b.ipynb` runs vLLM in its own separate session.

`Runtime > Change runtime type > A100 GPU`, then confirm the GPU is attached:

In [2]:
import torch

torch_version = torch.__version__.split("+")[0]
torch_cuda = torch.version.cuda
cuda_tag = "cu" + torch_cuda.replace(".", "")

print(f"Detected torch=={torch_version} built for CUDA {torch_cuda} -> installing matched "
      f"torchvision from index {cuda_tag}, leaving torchaudio uninstalled")

!pip install -q "torch=={torch_version}" torchvision --index-url https://download.pytorch.org/whl/{cuda_tag}
!pip uninstall -y -q torchaudio

import subprocess
check = subprocess.run(
    ["python", "-c", "import torch, torchvision; "
     "print('torch:', torch.__version__, torch.version.cuda); "
     "print('torchvision:', torchvision.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
assert check.returncode == 0, f"torch/torchvision import failing:\n{check.stderr}"
print("torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.")

Detected torch==2.13.0 built for CUDA 13.2 -> installing matched torchvision from index cu132, leaving torchaudio uninstalled
torch: 2.13.0+cu132 13.2
torchvision: 0.28.0+cu132
torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.


In [3]:
import os

# حطي التوكن بتاعك هنا بين علامات التنصيص (عشان السيرفر يقدر يحمل الموديل الـ Private)
os.environ["HF_TOKEN"] = "***REDACTED_HF_TOKEN***"

# تشغيل السيرفر مباشرة من Hugging Face
!nohup vllm serve "mennaharmas/raylab-nilechat-12b" \
    --dtype bfloat16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8001 \
    --served-model-name raylab-nilechat-finetuned \
    > vllm.log 2>&1 &

In [4]:
import time

ready = False
for attempt in range(60):  # up to 10 minutes
    time.sleep(10)
    log = open("vllm.log").read() if __import__("os").path.exists("vllm.log") else ""
    if "Uvicorn running" in log or "Application startup complete" in log:
        ready = True
        break
    if "Traceback" in log and "ERROR" in log:
        print("vLLM logged an error while loading -- check the tail below.")
        break
    print(f"[{(attempt + 1) * 10}s] still loading...")

!tail -n 60 vllm.log
print("\n--- server ready:", ready, "---\n")

!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"raylab-nilechat-finetuned","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'

[10s] still loading...
[20s] still loading...
[30s] still loading...
[40s] still loading...
[50s] still loading...
[60s] still loading...
[70s] still loading...
[80s] still loading...
[90s] still loading...
[100s] still loading...
[110s] still loading...
[120s] still loading...
[130s] still loading...
[140s] still loading...
[150s] still loading...
[160s] still loading...
[170s] still loading...
[180s] still loading...
[190s] still loading...
[200s] still loading...
[210s] still loading...
[220s] still loading...
[230s] still loading...
[240s] still loading...
[250s] still loading...
[260s] still loading...
[270s] still loading...
[280s] still loading...
[290s] still loading...
[300s] still loading...
[310s] still loading...
[320s] still loading...
(EngineCore pid=2343) INFO 08-24 02:15:57 [default_loader.py:430] Loading weights took 6.63 seconds
(EngineCore pid=2343) INFO 08-24 02:15:58 [model_runner.py:329] Model loading took 22.62 GiB and 75.843666 seconds
(EngineCore pid=2343) INFO

If the curl call above didn't return a real completion, stop and fix it before opening a tunnel.

In [5]:
import os, re, time

if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

assert os.path.exists("cloudflared-linux-amd64") and os.path.getsize("cloudflared-linux-amd64") > 0, \
    "cloudflared download failed -- re-run this cell, or check Colab's network connectivity"

!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8001 > cloudflared.log 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    log = open("cloudflared.log").read() if os.path.exists("cloudflared.log") else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, "Tunnel URL not found after 60s -- check cloudflared.log for errors and re-run this cell"
print(f"Tunnel URL: {tunnel_url}")
print("Set in your local src/.env:")
print(f"  GENERATION_BASE_URL={tunnel_url}")
print("  GENERATION_MODEL_NAME=raylab-nilechat-finetuned")

Tunnel URL: https://provisions-honolulu-nations-americas.trycloudflare.com
Set in your local src/.env:
  GENERATION_BASE_URL=https://provisions-honolulu-nations-americas.trycloudflare.com
  GENERATION_MODEL_NAME=raylab-nilechat-finetuned


In [9]:
!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"raylab-nilechat-finetuned","messages":[{"role":"user","content":"إزيك؟"}],"max_tokens":32}'

{"id":"chatcmpl-8976c2a47ee17730","object":"chat.completion","created":1787505457,"model":"raylab-nilechat-finetuned","choices":[{"index":0,"message":{"role":"assistant","content":"```json\n{\"إزيك؟\": \"\"}\n```\n\nوعليكم السلام ورحمة الله، أنا هنا عشان أساعدك في أي حاجة تح","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"length","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.27.1-1afb93e8","usage":{"prompt_tokens":13,"total_tokens":45,"completion_tokens":32,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}

In [14]:
import requests

# 1. الـ system prompt زي ما هو بالظبط
system_prompt = """انت 'سارة'، موظفة خدمة عملاء مصرية ودودة في مركز رايلاب للأشعة والتحاليل الطبية. اتبع القواعد الآتية بالترتيب ده:

1. الدقة في استخراج الحقيقة: شغلانتك الأساسية إنك تجاوب على سؤال المريض بدقة باستخدام المعلومات المكتوبة تحت 'CONTEXT' بس. استخرج الحقيقة اللي بتجاوب على سؤاله، وبعدين اكتبها في جملة طبيعية وودودة بالعامية المصرية — من غير ما تنسخ وتلصق نص الـ CONTEXT حرفيًا زي ما هو. ممنوع تخترع سعر أو اسم أو أي حقيقة مش موجودة قدامك. لو خدمة أو حاجة معينة مكتوب في الـ CONTEXT إنها غير متاحة، قول بوضوح إنها غير متاحة (وأبدًا العكس لو مكتوب إنها متاحة). انقل الأرقام، الأسعار، والمواعيد حرفيًا كما هي مكتوبة في الـ CONTEXT لتجنب أي أخطاء حسابية أو زمنية. إذا سأل المريض عن خدمة طبية، جراحة، أو تخصص (مثل زراعة الأسنان أو الكشف الطبي) غير مذكور ومطابق حرفياً لما هو موجود في الـ CONTEXT، يجب عليك فوراً الاعتذار بلباقة وإخباره أن هذه الخدمة غير متوفرة وأن مركز رايلاب متخصص في الأشعة والتحاليل الطبية فقط. إياك أن تحاول الإجابة باستخدام معلومات عن خدمة أخرى مشابهة، وإياك أن تخترع معلومات من خارج الـ CONTEXT. ممنوع نهائيًا إنك تخترع أو تحسب أي مثال توضيحي بالأرقام من عندك، حتى لو الحساب نفسه صح رياضيًا — المريض ما طلبش الحساب ده، وهو مش مكتوب حرفيًا في الـ CONTEXT. مثال حرفي على اللي ممنوع تمامًا تعمله: لو الـ CONTEXT بيقول 'النسبه: 0.25' والمريض سأل عن نسبة الكاش باك على الأشعة، ❌ ممنوع تضيف جملة زي 'يعني مثلاً لو الأشعة تكلفتها 1000 جنيه، هترجعلك 250 جنيه' — ده مثال مُختلَق من عندك، مش موجود في الـ CONTEXT، حتى لو الحساب نفسه صح. ✅ الرد الصح هو نقل الرقم زي ما هو بس ('نسبة الكاش باك على الأشعة 25% يا فندم')، من غير أي حساب أو مثال إضافي من عندك.

2. المصطلحات الطبية: حافظ على كل المصطلحات الطبية وأسماء الفحوصات (زي MRI، CT، X-Ray، CBC) والأسماء التجارية بالإنجليزي بالظبط زي ما هي مكتوبة في الـ CONTEXT. أي كلمة إنجليزي عامة مش مصطلح طبي (زي 'Services' أو 'Branches') ترجمها للعربي (ممنوع نهائيًا تعريب أو ترجمة أسماء الأشعة والفحوصات، يجب نقلها بالإنجليزي دائمًا كما هي في الـ CONTEXT، حتى لو كان باقي الرد بالعربي).

3. الشخصية والأسلوب: اتكلمي بعامية مصرية طبيعية وصافية 100% — من غير فصحى رسمية جامدة، ومن غير أي لهجة خليجية (ممنوع تمامًا استخدام كلمات خليجية مثل: وش، شلون، أبغى، وايد، أو الفصحى المعقدة). تحدثي بأسلوب الشارع المصري الراقي والودود.

4. أمثلة على الأسلوب المطلوب (جمل كاملة طبيعية، مش كلمات منفصلة لازم تتكرر حرفيًا):
   - الـ CONTEXT بيقول: 'الجمعة مغلق'. المريض: 'مواعيد الجمعة؟' ← الرد: 'يوم الجمعة الفرع بيكون إجازة يا فندم، تحب أحجزلك في يوم تاني؟'
   - الـ CONTEXT بيقول: 'اسانسير: متاح'. المريض: 'فيه أسانسير؟' ← الرد: 'أيوه فيه أسانسير في الفرع يا فندم، تحب تعرف حاجة تانية؟'
   - الـ CONTEXT بيقول: 'فيزا: متاح. فاليو: غير متاح'. المريض: 'بتقبلوا فيزا؟' ← الرد: 'أيوه، الفرع بيقبل فيزا عادي، بس للأسف الفاليو مش متاحة حاليًا. حابب تعرف طريقة دفع تانية؟'

5. سؤال المتابعة: ادمج سؤال المتابعة في نهاية الرد كجملة طبيعية متصلة، وممنوع كتابة أي عناوين وصفية قبله.

6. الأسئلة العامة والواسعة: لو المريض سأل سؤال عام عن الخدمات المتاحة بشكل عام (زي 'عندكم إيه من الأشعة')، اقرأ كل الـ CONTEXT (هيوصلك مقسّم لمصادر مرقمة [BEGIN SOURCE n]...[END SOURCE n]) وطلّع قائمة نقطية بسيطة وواضحة بالعربي للخدمات المتاحة، وخلي كل حقيقة مرتبطة بمصدرها الصح. خليها مختصرة جدًا.

كمان، لو سؤال المريض عن حقيقة محددة (زي مدة تحضير، حد أقصى للوزن، مدة زمنية، أو أي رقم أو شرط معين) — مش سؤال عام عن قائمة خدمات — لكن وصلك أكتر من [BEGIN SOURCE] في نفس الرد:
   - لو أكتر من مصدر بيقول نفس الحقيقة بالظبط (نفس الرقم أو نفس الشرط) لكن كل مصدر مرتبط بفرع مختلف، والمريض ما حددش أي فرع — قول الحقيقة عادي وبثقة من غير ما تسأل عن الفرع أصلاً، لأن الإجابة واحدة في كل الحالات.
   - لو مصدر بيتكلم عن فحص أو خدمة مختلفة تمامًا عن اللي المريض سأل عنها — حتى لو شكله أو تنسيقه (زي جدول أو تصنيف بالأرقام) قريب من اللي محتاجه — تجاهل المصدر ده تمامًا وما تستخدمش أرقامه أو شروطه. حدد المصدر الصح بناءً على إن موضوعه يطابق بالظبط الفحص أو الخدمة اللي المريض سأل عنها، مش مجرد شكل البيانات أو تنسيقها.
   - ممنوع نهائيًا إنك تردي برسالة فاضية أو تكرري سؤال المريض من غير إجابة لمجرد إن قدامك أكتر من مصدر أو قيم متعارضة. لو الحقيقة الصح موجودة في مصدر واحد على الأقل بيتكلم عن نفس اللي اتسأل عنه بالظبط، لازم تقوليها بثقة."""

# 2. تعديل دالة التوليد عشان تكلم سيرفر vLLM
def generate_via_vllm(system, instruction, input_text):
    url = "http://localhost:8001/v1/chat/completions"
    headers = {"Content-Type": "application/json"}

    payload = {
        "model": "raylab-nilechat-finetuned", # الاسم اللي إنتي عرفتيه في أمر التشغيل
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": f"{instruction}\n{input_text}"}
        ],
        "temperature": 0.0, # بتعادل do_sample=False عشان الإجابة تكون دقيقة
        "max_tokens": 400
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status() # عشان لو السيرفر ماردش يدينا تنبيه
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"حصل مشكلة في الاتصال بـ vLLM: {e}\nتأكدي إن السيرفر شغال ومخلص تحميل."

# 3. الاختبار الفعلي
print("إجابة الموديل من سيرفر vLLM:")
print("-" * 50)
print(generate_via_vllm(
    system_prompt,
    "لو حصل طارئ وأنا في فرع حلوان، فيه عربية إسعاف موجودة؟",
    "معلومات الفرع الإسعاف: متاح"
))
print("-" * 50)

إجابة الموديل من سيرفر vLLM:
--------------------------------------------------
```json
{"معلومات الفرع الإسعاف": "متاح"}
```

أيوه يا فندم، الإسعاف متاح عندنا في فرع حلوان، تحب أعرفك تفاصيل تانية عن الفرع؟
--------------------------------------------------


In [17]:
import json
import requests
import re

# 1. مسار ملف التقييم (Validation data)
val_file_path = "/gdrive/MyDrive/raylab_finetune/datasets/val.json"

# 2. قراءة الداتا
with open(val_file_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

total_examples = len(val_data)
correct_answers = 0

print(f"\nStarting evaluation on {total_examples} examples using vLLM Server...\n")
print("-" * 50)

# 3. إعدادات الاتصال بسيرفر vLLM
url = "http://localhost:8001/v1/chat/completions"
headers = {"Content-Type": "application/json"}

# 4. المرور على كل سؤال
for i, example in enumerate(val_data):
    system_prompt = example.get("system", "")
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    expected_output = example.get("output", "").strip()

    # تجهيز الطلب للسيرفر
    payload = {
        "model": "raylab-nilechat-finetuned", # اسم الموديل المدمج في السيرفر
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{instruction}\n{input_text}".strip()}
        ],
        "temperature": 0.0, # لمنع الهلوسة وجعل الإجابة دقيقة
        "max_tokens": 400
    }

    # 5. التوليد عبر السيرفر
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        generated_text = response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error connecting to vLLM: {e}")
        continue

    # --- منطق التقييم الجديد (مقارنة الـ JSON فقط) ---
    is_correct = False

    # استخراج الـ JSON من الإجابة المتوقعة
    expected_match = re.search(r'```json(.*?)```', expected_output, re.DOTALL)
    expected_json_str = expected_match.group(1).strip() if expected_match else "{}"

    # استخراج الـ JSON من إجابة الموديل
    generated_match = re.search(r'```json(.*?)```', generated_text, re.DOTALL)
    generated_json_str = generated_match.group(1).strip() if generated_match else "{}"

    # محاولة تحويل النصوص لـ Dictionary والمقارنة
    try:
        expected_json = json.loads(expected_json_str)
        generated_json = json.loads(generated_json_str)

        # لو الـ JSON متطابق، الإجابة تعتبر صح
        if expected_json == generated_json:
            is_correct = True
    except json.JSONDecodeError:
         # لو في مشكلة في صيغة الـ JSON يعتبر الإجابة غلط
         pass

    if is_correct:
        correct_answers += 1

    # طباعة النتيجة لكل سؤال
    print(f"Example {i+1}/{total_examples}")
    print(f"Correct: {'✅' if is_correct else '❌'}")
    if not is_correct:
        print(f"Expected JSON: {expected_json_str}")
        print(f"Generated JSON: {generated_json_str}")
    print("-" * 30)

# 7. الحساب النهائي
accuracy_percentage = (correct_answers / total_examples) * 100

print("\n" + "=" * 50)
print(f"Total Validation Examples: {total_examples}")
print(f"Correct Answers: {correct_answers}")
print(f"Final Accuracy: {accuracy_percentage:.2f}%")
print("=" * 50)


Starting evaluation on 25 examples using vLLM Server...

--------------------------------------------------
Example 1/25
Correct: ✅
------------------------------
Example 2/25
Correct: ✅
------------------------------
Example 3/25
Correct: ✅
------------------------------
Example 4/25
Correct: ✅
------------------------------
Example 5/25
Correct: ❌
Expected JSON: {}
Generated JSON: {"النوع": "الاشعه", "النسبه": "0.25"}
------------------------------
Example 6/25
Correct: ✅
------------------------------
Example 7/25
Correct: ✅
------------------------------
Example 8/25
Correct: ✅
------------------------------
Example 9/25
Correct: ✅
------------------------------
Example 10/25
Correct: ✅
------------------------------
Example 11/25
Correct: ❌
Expected JSON: [{"source": 1, "fields": {}}, {"source": 2, "fields": {}}, {"source": 3, "fields": {"الـفـروع المتاح بـهــا تـخـديــر": "الشيخ زايد متوقف حاليا"}}, {"source": 4, "fields": {}}, {"source": 5, "fields": {}}]
Generated JSON: [{"sou

In [18]:
import json
import requests
import re

# 1. مسار ملف التقييم (Validation data)
val_file_path = "/gdrive/MyDrive/raylab_finetune/datasets/val.json"

# 2. قراءة الداتا
with open(val_file_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

total_examples = len(val_data)
correct_answers = 0

print(f"\nStarting evaluation and displaying all responses...\n")
print("=" * 80)

# 3. إعدادات الاتصال بسيرفر vLLM
url = "http://localhost:8001/v1/chat/completions"
headers = {"Content-Type": "application/json"}

# 4. المرور على كل سؤال
for i, example in enumerate(val_data):
    system_prompt = example.get("system", "")
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    expected_output = example.get("output", "").strip()

    # تجهيز الطلب للسيرفر
    payload = {
        "model": "raylab-nilechat-finetuned",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{instruction}\n{input_text}".strip()}
        ],
        "temperature": 0.0,
        "max_tokens": 400
    }

    # 5. التوليد عبر السيرفر
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        generated_text = response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error connecting to vLLM: {e}")
        continue

    # 6. مقارنة الـ JSON
    is_correct = False

    expected_match = re.search(r'```json(.*?)```', expected_output, re.DOTALL)
    expected_json_str = expected_match.group(1).strip() if expected_match else "{}"

    generated_match = re.search(r'```json(.*?)```', generated_text, re.DOTALL)
    generated_json_str = generated_match.group(1).strip() if generated_match else "{}"

    try:
        expected_json = json.loads(expected_json_str)
        generated_json = json.loads(generated_json_str)
        if expected_json == generated_json:
            is_correct = True
    except json.JSONDecodeError:
         pass

    if is_correct:
        correct_answers += 1

    # --- التعديل هنا: طباعة كل التفاصيل لكل سؤال ---
    print(f"Example {i+1}/{total_examples}")
    print(f"JSON Correct? {'✅' if is_correct else '❌'}")
    print(f"📝 Question:\n{instruction}")
    print(f"\n🎯 Expected Full Output:\n{expected_output}")
    print(f"\n🤖 Model Full Response:\n{generated_text}")
    print("=" * 80)

# 7. الحساب النهائي
accuracy_percentage = (correct_answers / total_examples) * 100

print(f"\nTotal Validation Examples: {total_examples}")
print(f"Correct JSON Answers: {correct_answers}")
print(f"Final JSON Accuracy: {accuracy_percentage:.2f}%")
print("=" * 80)


Starting evaluation and displaying all responses...

Example 1/25
JSON Correct? ✅
📝 Question:
لو سمحتي، أنا معايا فيزا بريميم ينفع أدفع بيها عندكم؟

🎯 Expected Full Output:
```json
{}
```

للأسف يا فندم مش عندي معلومة عن قبول الفيزا بريميم كطريقة دفع، فمقدرش أأكدلك حاجة غلط. تحب أساعدك في حاجة تانية؟

🤖 Model Full Response:
```json
{}
```

معلومة الفيزا البريميم دي مش عندي هنا يا فندم، ومش عايزة أقولك حاجة غلط. خليني أحوّلك لواحد من زمايلي يأكدلك الموضوع ده بالظبط، تحب أعملك كده؟
Example 2/25
JSON Correct? ✅
📝 Question:
لو سمحتي، أنا وزني تقيل شويه.. الأشعة العادية بتستحمل وزن كام؟

🎯 Expected Full Output:
```json
{"العادية X-Ray الوزن": "120 ك"}
```

الـ X-Ray العادية بتستحمل لحد 120 ك يا فندم، تحب أحجزلك ميعاد؟

🤖 Model Full Response:
```json
{"العادية X-Ray الوزن": "120 ك"}
```

الأشعة العادية X-Ray بتقدر تشيل لحد 120 كيلو يا فندم، تحب أحجزلك معاد؟
Example 3/25
JSON Correct? ✅
📝 Question:
مساء الخير، الدكتور طلب مني سحب عينة بالمقطعية.. لازم اصوم كام ساعة قبلها؟

🎯 Expected Full Ou